In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")

SOURCE = f"/Volumes/{CATALOG}/raw/source_olist"
SCHEMA_CIBLE = f"{CATALOG}.lakehouse"
print(f"source : {SOURCE}\ncible  : {SCHEMA_CIBLE}")

In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.insert(0, os.path.abspath(".."))   # src/jobs/.. = src/

from olist.bronze import FICHIERS_SOURCE, LIGNES_ATTENDUES, lire_csv_brut, ecrire_table, controler_table

In [0]:
for nom, fichier in FICHIERS_SOURCE.items():
    table = f"{SCHEMA_CIBLE}.bronze_{nom}"
    ecrire_table(lire_csv_brut(spark, f"{SOURCE}/{fichier}"), table)
    print(f"écrite : {table}")

In [0]:
erreurs = {}
for nom, attendu in LIGNES_ATTENDUES.items():
    table = f"{SCHEMA_CIBLE}.bronze_{nom}"
    problemes = controler_table(spark, table, attendu)
    print(f"{table:<48} {'OK' if not problemes else 'ERREUR : ' + ' ; '.join(problemes)}")
    if problemes:
        erreurs[nom] = problemes

assert not erreurs, f"Bronze incorrect : {erreurs}"
print("\nBronze validé : 9 tables conformes.")

In [0]:
display(spark.table(f"{SCHEMA_CIBLE}.bronze_orders").limit(5))